# Remove redundant generated datasets

User-requested cleanup. Keep raw data, current32 features, frozen folds/labels/candidates, current model OOF/test predictions, fitted weights and compact historical summaries. Old caches and predictions can be regenerated by the original notebooks.

In [1]:
from pathlib import Path
import json

redundant = ['outputs/candidates/ranker_candidates.csv', 'outputs/candidates/ranker_groups.csv', 'outputs/features/features.parquet', 'outputs/student_test/features.parquet', 'results/xgb_ranker/20261118_121050/oof_xgb_ranker.csv', 'results/step7_xgb/untuned/oof_untuned.csv', 'results/step7_xgb/tuned_sample10000/oof_tuned_sample10000.csv', 'results/exclude_33_54/oof_xgb_ranker.csv', 'results/step7_xgb/untuned/student_test_predictions.csv', 'results/step7_xgb/tuned_sample10000/student_test_predictions.csv', 'results/exclude_33_54/student_test_predictions.csv', 'results/hmm/ranker/candidate_probabilities.parquet']
removed = []
for name in redundant:
    path = Path(name)
    if path.is_file():
        removed.append({"path": name, "bytes": path.stat().st_size})
        path.unlink()
manifest = Path("outputs/inventory/redundant_cleanup.json")
previous = json.loads(manifest.read_text()) if manifest.exists() else {"removed": []}
known = {row["path"]: row for row in previous["removed"]}
known.update({row["path"]: row for row in removed})
record = {"removed": list(known.values()), "bytes_removed": sum(row["bytes"] for row in known.values()),
          "kept": "raw data; current32 features and base-model OOF; fitted models; labels/folds/canonical candidates; historical summaries",
          "regeneration": "Run historical notebooks from feature generation onward to rebuild removed historical predictions."}
manifest.write_text(json.dumps(record, indent=2) + "\n")
print("Removed", len(removed), "files; cumulative GB:", round(record["bytes_removed"] / 1e9, 2))


Removed 12 redundant files; 3.20 GB total.
